In [1]:
cd ..

/home/jovyan/Robbi/dea-intertidal


In [ ]:
pip install -r requirements.in --quiet

In [ ]:
pip install eo-tides==0.6.3

In [ ]:
%load_ext autoreload
%autoreload 2

import datacube
from intertidal.io import load_data
from dea_tools.dask import create_local_dask_cluster

dc = datacube.Datacube()

client = create_local_dask_cluster()

In [ ]:
# Load satellite data and dataset IDs for metadata
satellite_ds, dss_s2, dss_ls = load_data(
    dc=dc,
    study_area="x150y098",
    # geom=geom,
    time_range=("2021", "2023"),
    resolution=10,
    crs="EPSG:3577",
    include_s2=True,
    include_ls=False,
    filter_gqa=True,
    max_cloudcover=90,
    skip_broken_datasets=True,
    dataset_maturity="final",
)

# No Landsat
band = satellite_ds.ndwi.compute()
band.notnull().sum(dim="time").plot.imshow()

In [ ]:
# Load satellite data and dataset IDs for metadata
satellite_ds, dss_s2, dss_ls = load_data(
    dc=dc,
    study_area="x150y098",
    # geom=geom,
    time_range=("2021", "2023"),
    resolution=10,
    crs="EPSG:3577",
    include_s2=True,
    include_ls=False,
    filter_gqa=False,
    max_cloudcover=90,
    skip_broken_datasets=True,
    dataset_maturity="final",
)

# No Landsat, no QGA
band = satellite_ds.ndwi.compute()
band.notnull().sum(dim="time").plot.imshow()

In [ ]:
import pandas as pd
(pd.Series([i.metadata.search_fields["gqa_iterative_mean_xy"] for i in dss_s2]) > 20).mean()

In [ ]:
pd.Series([i.metadata.search_fields["gqa_iterative_mean_xy"] for i in dss_s2]).quantile(0.95)

In [ ]:
# Load satellite data and dataset IDs for metadata
satellite_ds, dss_s2, dss_ls = load_data(
    dc=dc,
    study_area="x150y098",
    # geom=geom,
    time_range=("2021", "2023"),
    resolution=10,
    crs="EPSG:3577",
    include_s2=False,
    include_ls=True,
    filter_gqa=True,
    max_cloudcover=90,
    skip_broken_datasets=True,
    dataset_maturity="final",
)

# No Sentinel-2
band = satellite_ds.ndwi.compute()
band.notnull().sum(dim="time").plot.imshow()

In [ ]:
# Load satellite data and dataset IDs for metadata
satellite_ds, dss_s2, dss_ls = load_data(
    dc=dc,
    study_area="x150y098",
    # geom=geom,
    time_range=("2021", "2023"),
    resolution=10,
    crs="EPSG:3577",
    include_s2=True,
    include_ls=True,
    filter_gqa=True,
    max_cloudcover=90,
    skip_broken_datasets=True,
    dataset_maturity="final",
)

# Both
band = satellite_ds.ndwi.compute()
band.notnull().sum(dim="time").plot.imshow()

In [3]:
import subprocess

# Remote COGs as vsicurl paths (GDAL can stream directly)
urls = [
    "/vsicurl/https://dea-public-data-dev.s3-ap-southeast-2.amazonaws.com/derivative/ga_s2_tidal_composites_cyear_3/0-0-2/continental_mosaics/2022--P1Y/ga_s2_tidal_composites_cyear_3_2022_low-swir-2.tif",
    "/vsicurl/https://dea-public-data-dev.s3-ap-southeast-2.amazonaws.com/derivative/ga_s2_tidal_composites_cyear_3/0-0-2/continental_mosaics/2022--P1Y/ga_s2_tidal_composites_cyear_3_2022_low-nir-1.tif",
    "/vsicurl/https://dea-public-data-dev.s3-ap-southeast-2.amazonaws.com/derivative/ga_s2_tidal_composites_cyear_3/0-0-2/continental_mosaics/2022--P1Y/ga_s2_tidal_composites_cyear_3_2022_low-green.tif",
]

# Write to a temporary list file
with open("cog_vsicurl_list.txt", "w") as f:
    f.write("\n".join(urls))

# Create VRT
subprocess.run([
    "gdalbuildvrt",
    "-input_file_list", "cog_vsicurl_list.txt",
    "-separate",
    # "-oo", "USE_PATHS_AS_VSICURL=YES",  # Treat input paths as remote
    "-allow_projection_difference",     # Avoids reprojection overhead
    "tidal_composite_low_false.vrt"
], check=True)

print("✅ VRT created using /vsicurl")


0...10...20...30...40...50...60...70...80...90...100 - done.
✅ VRT created using /vsicurl


In [4]:
import subprocess

# Remote COGs as vsicurl paths (GDAL can stream directly)
urls = [
    "/vsicurl/https://dea-public-data-dev.s3-ap-southeast-2.amazonaws.com/derivative/ga_s2_tidal_composites_cyear_3/0-0-2/continental_mosaics/2022--P1Y/ga_s2_tidal_composites_cyear_3_2022_high-swir-2.tif",
    "/vsicurl/https://dea-public-data-dev.s3-ap-southeast-2.amazonaws.com/derivative/ga_s2_tidal_composites_cyear_3/0-0-2/continental_mosaics/2022--P1Y/ga_s2_tidal_composites_cyear_3_2022_high-nir-1.tif",
    "/vsicurl/https://dea-public-data-dev.s3-ap-southeast-2.amazonaws.com/derivative/ga_s2_tidal_composites_cyear_3/0-0-2/continental_mosaics/2022--P1Y/ga_s2_tidal_composites_cyear_3_2022_high-green.tif",
]

# Write to a temporary list file
with open("cog_vsicurl_list.txt", "w") as f:
    f.write("\n".join(urls))

# Create VRT
subprocess.run([
    "gdalbuildvrt",
    "-input_file_list", "cog_vsicurl_list.txt",
    "-separate",
    # "-oo", "USE_PATHS_AS_VSICURL=YES",  # Treat input paths as remote
    "-allow_projection_difference",     # Avoids reprojection overhead
    "tidal_composite_high_false.vrt"
], check=True)

print("✅ VRT created using /vsicurl")

0...10...20...30...40...50...60...70...80...90...100 - done.
✅ VRT created using /vsicurl


In [5]:
import subprocess

# Remote COGs as vsicurl paths (GDAL can stream directly)
urls = [
    "/vsicurl/https://dea-public-data-dev.s3-ap-southeast-2.amazonaws.com/derivative/ga_s2_tidal_composites_cyear_3/0-0-2/continental_mosaics/2022--P1Y/ga_s2_tidal_composites_cyear_3_2022_low-red.tif",
    "/vsicurl/https://dea-public-data-dev.s3-ap-southeast-2.amazonaws.com/derivative/ga_s2_tidal_composites_cyear_3/0-0-2/continental_mosaics/2022--P1Y/ga_s2_tidal_composites_cyear_3_2022_low-green.tif",
    "/vsicurl/https://dea-public-data-dev.s3-ap-southeast-2.amazonaws.com/derivative/ga_s2_tidal_composites_cyear_3/0-0-2/continental_mosaics/2022--P1Y/ga_s2_tidal_composites_cyear_3_2022_low-blue.tif",
]

# Write to a temporary list file
with open("cog_vsicurl_list.txt", "w") as f:
    f.write("\n".join(urls))

# Create VRT
subprocess.run([
    "gdalbuildvrt",
    "-input_file_list", "cog_vsicurl_list.txt",
    "-separate",
    # "-oo", "USE_PATHS_AS_VSICURL=YES",  # Treat input paths as remote
    "-allow_projection_difference",     # Avoids reprojection overhead
    "tidal_composite_low_rgb.vrt"
], check=True)

print("✅ VRT created using /vsicurl")

0...10...20...30...40...50...60...70...80...90...100 - done.
✅ VRT created using /vsicurl


In [4]:
import subprocess

version = "0-0-5"
tide = "high"

# Remote COGs as vsicurl paths (GDAL can stream directly)
urls = [
    f"/vsicurl/https://dea-public-data-dev.s3-ap-southeast-2.amazonaws.com/derivative/ga_s2_tidal_composites_cyear_3/{version}/continental_mosaics/2022--P1Y/ga_s2_tidal_composites_cyear_3_2022_{tide}-red.tif",
    f"/vsicurl/https://dea-public-data-dev.s3-ap-southeast-2.amazonaws.com/derivative/ga_s2_tidal_composites_cyear_3/{version}/continental_mosaics/2022--P1Y/ga_s2_tidal_composites_cyear_3_2022_{tide}-green.tif",
    f"/vsicurl/https://dea-public-data-dev.s3-ap-southeast-2.amazonaws.com/derivative/ga_s2_tidal_composites_cyear_3/{version}/continental_mosaics/2022--P1Y/ga_s2_tidal_composites_cyear_3_2022_{tide}-blue.tif",
]

# Write to a temporary list file
with open("cog_vsicurl_list.txt", "w") as f:
    f.write("\n".join(urls))

# Create VRT
subprocess.run([
    "gdalbuildvrt",
    "-input_file_list", "cog_vsicurl_list.txt",
    "-separate",
    "-allow_projection_difference",     # Avoids reprojection overhead
    f"tidal_composite_{version}_{tide}_rgb.vrt"
], check=True)


0...10...20...30...40...50...60...70...80...90...100 - done.
✅ VRT created using /vsicurl


In [4]:
import subprocess

version = "0-0-10"
tide = "high"

# Create VRT
subprocess.run(
    [
        "gdalbuildvrt",
        "-separate",
        "-allow_projection_difference",  # Avoids reprojection overhead
        f"tidal_composite_{version}_{tide}_rgb.vrt",
        f"/vsicurl/https://dea-public-data-dev.s3-ap-southeast-2.amazonaws.com/derivative/ga_s2_tidal_composites_cyear_3/{version}/continental_mosaics/2022--P1Y/ga_s2_tidal_composites_cyear_3_2022_{tide}-red.tif",
        f"/vsicurl/https://dea-public-data-dev.s3-ap-southeast-2.amazonaws.com/derivative/ga_s2_tidal_composites_cyear_3/{version}/continental_mosaics/2022--P1Y/ga_s2_tidal_composites_cyear_3_2022_{tide}-green.tif",
        f"/vsicurl/https://dea-public-data-dev.s3-ap-southeast-2.amazonaws.com/derivative/ga_s2_tidal_composites_cyear_3/{version}/continental_mosaics/2022--P1Y/ga_s2_tidal_composites_cyear_3_2022_{tide}-blue.tif",
    ],
    check=True,
)

0...10...20...30...40...50...60...70...80...90...100 - done.


Warning 1: Can't open /vsicurl/https://dea-public-data-dev.s3-ap-southeast-2.amazonaws.com/derivative/ga_s2_tidal_composites_cyear_3/0-0-10/continental_mosaics/2022--P1Y/ga_s2_tidal_composites_cyear_3_2022_high-green.tif. Skipping it


CompletedProcess(args=['gdalbuildvrt', '-separate', '-allow_projection_difference', 'tidal_composite_0-0-10_high_rgb.vrt', '/vsicurl/https://dea-public-data-dev.s3-ap-southeast-2.amazonaws.com/derivative/ga_s2_tidal_composites_cyear_3/0-0-10/continental_mosaics/2022--P1Y/ga_s2_tidal_composites_cyear_3_2022_high-red.tif', '/vsicurl/https://dea-public-data-dev.s3-ap-southeast-2.amazonaws.com/derivative/ga_s2_tidal_composites_cyear_3/0-0-10/continental_mosaics/2022--P1Y/ga_s2_tidal_composites_cyear_3_2022_high-green.tif', '/vsicurl/https://dea-public-data-dev.s3-ap-southeast-2.amazonaws.com/derivative/ga_s2_tidal_composites_cyear_3/0-0-10/continental_mosaics/2022--P1Y/ga_s2_tidal_composites_cyear_3_2022_high-blue.tif'], returncode=0)

In [17]:
import xarray as xr

xr.DataArray([0, 1, 2, 3, 4, 5, 5]).rank(dim="dim_0")

<xarray.DataArray (dim_0: 7)> Size: 56B
array([1. , 2. , 3. , 4. , 5. , 6.5, 6.5])
Dimensions without coordinates: dim_0

In [ ]:
# Triggers stats calculation for all bands in-place
subprocess.run(["gdalinfo", "-approx_stats", "combined_tidal_composite.vrt"], check=True)

In [3]:
!eo3-validate \
../metadata/ga_s2_tidal_composites_cyear_3.odc-product.yaml \
../metadata/eo3_intertidal.odc-type.yaml \
../data/processed/ga_s2_tidal_composites_cyear_3/testing/tes/ting/2019--P1Y/ga_s2_tidal_composites_cyear_3_testing_2019--P1Y_final.odc-metadata.yaml \
--thorough

✓ file:///home/jovyan/Robbi/dea-intertidal/metadata/ga_s2_tidal_composites_cyear_3.odc-product.yaml
✓ file:///home/jovyan/Robbi/dea-intertidal/metadata/eo3_intertidal.odc-type.yaml
✓ file:///home/jovyan/Robbi/dea-intertidal/data/processed/ga_s2_tidal_composites_cyear_3/testing/tes/ting/2019--P1Y/ga_s2_tidal_composites_cyear_3_testing_2019--P1Y_final.odc-metadata.yaml

valid: 3 paths


In [ ]:
!gdalinfo ../data/processed/ga_s2_tidal_composites_cyear_3/testing/tes/ting/2019--P1Y/ga_s2_tidal_composites_cyear_3_testing_2019--P1Y_final_high-nir-2.tif